In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [8]:
import os, pandas as pd, numpy as np, torch, torch.nn as nn, torch.optim as optim, cv2, matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from torchvision.models import densenet121, DenseNet121_Weights
from PIL import Image
from glob import glob
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import warnings

warnings.filterwarnings("ignore", message=".*iCCP: profile 'ICC Profile'.*")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = '/kaggle/input/datasets/organizations/nih-chest-xrays/data/'
CSV_PATH = os.path.join(DATA_DIR, 'Data_Entry_2017.csv')
disease_labels = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 
                  'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Pleural_Thickening']

all_image_paths = {os.path.basename(x): x for x in glob(os.path.join(DATA_DIR, 'images_*/images/*.png'))}

In [9]:
import os, pandas as pd, numpy as np, torch, torch.nn as nn, torch.optim as optim, cv2
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import densenet121, DenseNet121_Weights
from PIL import Image
from glob import glob
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = '/kaggle/input/data/'
all_image_paths = {os.path.basename(x): x for x in glob(os.path.join(DATA_DIR, 'images_/images/.png'))}

disease_labels = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Pleural_Thickening']

In [11]:
import os
import pandas as pd
import numpy as np
import torch
from glob import glob
from sklearn.model_selection import train_test_split
from torch.utils.data import WeightedRandomSampler

# 1. DYNAMIC PATH SEARCHING (Fixes FileNotFoundError)
# This looks for the CSV file anywhere in the /kaggle/input/ directory
csv_search = glob('/kaggle/input/**/Data_Entry_2017.csv', recursive=True)
if not csv_search:
    raise FileNotFoundError("Could not find Data_Entry_2017.csv. Please ensure the dataset is added.")

CSV_PATH = csv_search[0]
DATA_DIR = os.path.dirname(CSV_PATH)
print(f"Found CSV at: {CSV_PATH}")

# 2. IMAGE PATH MAPPING
# Maps filename (e.g., '00000001_000.png') to its full system path
all_image_paths = {os.path.basename(x): x for x in glob('/kaggle/input/**/*.png', recursive=True)}
print(f"Found {len(all_image_paths)} images.")

# 3. LOAD AND FILTER DATA
df = pd.read_csv(CSV_PATH)
# Keep only rows where the actual image file exists on disk
df = df[df['Image Index'].isin(all_image_paths.keys())].reset_index(drop=True)

# 4. MULTI-LABEL EXPANDING (Fixes KeyError)
disease_labels = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 
                  'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Pleural_Thickening']

for disease in disease_labels:
    df[disease] = df['Finding Labels'].apply(lambda x: 1 if disease in x else 0)

# 5. TRAIN/VAL SPLIT
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# 6. BALANCED SAMPLER LOGIC (Fixes 54% Imbalance)
# We calculate weights so the model sees rare diseases as often as "No Finding"
class_counts = train_df[disease_labels].sum(axis=0).values
class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)

sample_weights = []
for _, row in train_df[disease_labels].iterrows():
    # Assign image weight based on its rarest disease label
    current_weights = row.values * class_weights.numpy()
    weight = current_weights[current_weights > 0].max() if current_weights.max() > 0 else 1.0/len(train_df)
    sample_weights.append(weight)

sampler = WeightedRandomSampler(
    weights=sample_weights, 
    num_samples=len(sample_weights), 
    replacement=True
)

print(f"Data ready. Training: {len(train_df)} | Validation: {len(val_df)}")

Found CSV at: /kaggle/input/datasets/organizations/nih-chest-xrays/data/Data_Entry_2017.csv
Found 112120 images.
Data ready. Training: 89696 | Validation: 22424


In [8]:
class_counts = train_df[disease_labels].sum(axis=0).values
class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)

sample_weights = []
for _, row in train_df[disease_labels].iterrows():
    
    weight = (row.values * class_weights.numpy()).max()
    sample_weights.append(weight)

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

In [12]:
class SehaTrackDataset(Dataset):
    
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['Image Index']
        img_path = all_image_paths[img_name]
        
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        img = clahe.apply(img)
        
        img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_GRAY2RGB))
        labels = self.df.iloc[idx][disease_labels].values.astype('float32')
        
        if self.transform:
            img = self.transform(img)
        
        return img, torch.tensor(labels)


In [13]:
data_transforms = transforms.Compose([
transforms.Resize((224, 224)),
transforms.RandomHorizontalFlip(),
transforms.ToTensor(),
transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_loader = DataLoader(SehaTrackDataset(train_df, data_transforms), batch_size=32, sampler=sampler)
val_loader = DataLoader(SehaTrackDataset(val_df, data_transforms), batch_size=32, shuffle=False)

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import densenet121, DenseNet121_Weights

# Define Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, pos_weights=None):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.pos_weights = pos_weights
        self.bce = nn.BCEWithLogitsLoss(reduction='none', pos_weight=self.pos_weights)

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss)
        return (self.alpha * (1 - pt) ** self.gamma * bce_loss).mean()

loss_weights = torch.tensor([2.0, 3.0, 1.5, 1.0, 2.5, 2.0, 10.0, 2.0, 5.0, 5.0]).to(device)

model = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
model.classifier = nn.Linear(model.classifier.in_features, len(disease_labels))
model = model.to(device)

criterion = FocalLoss(pos_weights=loss_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [15]:
import torch
from tqdm import tqdm

def train_sehatrack(epochs=5):
    for epoch in range(epochs):
        model.train()
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        
        for imgs, lbls in loop:
            imgs, lbls = imgs.to(device), lbls.to(device)
            
            optimizer.zero_grad()
            loss = criterion(model(imgs), lbls)
            loss.backward()
            optimizer.step()
            
            loop.set_postfix(loss=loss.item())
        
        print(f"Epoch {epoch+1} finished.")
    
    torch.save(model.state_dict(), 'sehatrack_final.pth')

# Run training
train_sehatrack(epochs=5)


Epoch 1: 100%|██████████| 2803/2803 [54:19<00:00,  1.16s/it, loss=0.202]


Epoch 1 finished.


Epoch 2: 100%|██████████| 2803/2803 [53:02<00:00,  1.14s/it, loss=0.121] 


Epoch 2 finished.


Epoch 3: 100%|██████████| 2803/2803 [53:18<00:00,  1.14s/it, loss=0.127] 


Epoch 3 finished.


Epoch 4: 100%|██████████| 2803/2803 [53:12<00:00,  1.14s/it, loss=0.105] 


Epoch 4 finished.


Epoch 5: 100%|██████████| 2803/2803 [52:43<00:00,  1.13s/it, loss=0.0739]

Epoch 5 finished.


In [17]:
import torch
import numpy as np
from sklearn.metrics import classification_report

def generate_final_report(model, loader, save_path="final_report.txt"):
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outputs = model(imgs)
            preds = torch.sigmoid(outputs).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(lbls.cpu().numpy())

    # Convert to numpy arrays
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    # Binarize predictions at threshold 0.5
    bin_preds = (all_preds >= 0.5).astype(int)

    # Generate classification report
    report = classification_report(all_labels, bin_preds, target_names=disease_labels)

    # Print to console
    print(report)

    # Save to file
    with open(save_path, "w") as f:
        f.write(report)

    return report

# Example usage:
final_report = generate_final_report(model, val_loader, save_path="sehatrack_report.txt")


                    precision    recall  f1-score   support

       Atelectasis       0.19      0.57      0.28      2225
      Cardiomegaly       0.16      0.57      0.25       557
          Effusion       0.36      0.60      0.45      2625
      Infiltration       0.27      0.30      0.28      3951
              Mass       0.15      0.55      0.23      1141
            Nodule       0.09      0.47      0.16      1285
         Pneumonia       0.03      0.11      0.05       294
      Pneumothorax       0.30      0.52      0.38      1046
     Consolidation       0.10      0.31      0.15       894
Pleural_Thickening       0.09      0.28      0.13       663

         micro avg       0.18      0.45      0.26     14681
         macro avg       0.17      0.43      0.24     14681
      weighted avg       0.22      0.45      0.28     14681
       samples avg       0.16      0.20      0.17     14681



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
import torch
import torch.nn as nn
import numpy as np
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

# 1. DEFINE DEVICE (Fixes your NameError)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. DEFINE LABELS
disease_labels = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 
                  'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Pleural_Thickening']

# 3. DEFINE FOCAL LOSS CLASS
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, pos_weights=None):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.pos_weights = pos_weights
        self.bce = nn.BCEWithLogitsLoss(reduction='none', pos_weight=self.pos_weights)

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss) 
        focal_loss = self.alpha * (1 - pt)**self.gamma * bce_loss
        return focal_loss.mean()

# 4. DEFINE GRAD-CAM WRAPPER
class GradCAMModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.gradients = None
        self.activations = None

    def activations_hook(self, grad):
        self.gradients = grad

    def forward(self, x):
        x = self.model.features(x)
        h = x.register_hook(self.activations_hook) # Register hook for visualization
        self.activations = x
        x = self.model.avgpool(x)
        x = self.model.classifier(x)
        return x

    def get_activations_gradient(self):
        return self.gradients

    def get_activations(self, x):
        return self.activations

# 5. INITIALIZE CONVNEXT MODEL
# Using ConvNeXt-Tiny for high-performance medical texture recognition
raw_model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)

# Modify classifier for 10 diseases
n_inputs = raw_model.classifier[2].in_features
raw_model.classifier[2] = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(n_inputs, len(disease_labels))
)

# Wrap and move to device
model = GradCAMModel(raw_model).to(device)

# 6. LOSS AND OPTIMIZER
# High weight (10.0) for Pneumonia at index 6 to improve recall
loss_weights = torch.tensor([2.0, 3.0, 1.5, 1.0, 2.5, 2.0, 10.0, 2.0, 5.0, 5.0]).to(device)
criterion = FocalLoss(pos_weights=loss_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

print(f"Setup Complete. Running on: {device}")

Setup Complete. Running on: cuda


In [15]:
import torch
from tqdm import tqdm
def train_sehatrack(epochs=5):
    history = {'train_loss': [], 'val_loss': []}
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        loop = tqdm(train_loader)
        for imgs, lbls in loop:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), lbls)
            loss.backward(); optimizer.step()
            train_loss += loss.item()
            loop.set_description(f"Epoch {epoch+1}")
    
    # Validation Phase
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            val_loss += criterion(model(imgs.to(device)), lbls.to(device)).item()
    
    avg_train = train_loss / len(train_loader)
    avg_val = val_loss / len(val_loader)
    history['train_loss'].append(avg_train)
    history['val_loss'].append(avg_val)
    
    print(f"Epoch {epoch+1}: Train Loss {avg_train:.4f} | Val Loss {avg_val:.4f}")
    scheduler.step(avg_val)
    torch.save(model.state_dict(), 'sehatrack_final.pth')
    return history
history = train_sehatrack(epochs=5)

Epoch 5: 100%|██████████| 2803/2803 [1:02:50<00:00,  1.35s/it]


RuntimeError: cannot register a hook on a tensor that doesn't require gradient

In [2]:
plt.figure(figsize=(12, 5))
plt.plot(history['train_loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('SehaTrack Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
def visualize_results(model, dataset, idx=0):
    
    model.eval()
    img_tensor, labels = dataset[idx]
# Generate Grad-CAM
input_batch = img_tensor.unsqueeze(0).to(device)
features = model.features(input_batch) # For ConvNeXt
output = model(input_batch)

# Get the heatmap for the highest predicted class
pred_idx = torch.argmax(output).item()
target = output[:, pred_idx]
target.backward()

gradients = model.get_act_grads() # Requires a hook, standard Grad-CAM logic
# (Note: For brevity, ensure you have a standard Grad-CAM hook set up)

# Displaying images side-by-side
fig, ax = plt.subplots(1, 3, figsize=(20, 10))

# 1. Original Image
raw_img = cv2.imread(all_image_paths[dataset.df.iloc[idx]['Image Index']])
ax[0].imshow(cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB))
ax[0].set_title("Original X-Ray")

# 2. CLAHE Image
gray = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(gray)
ax[1].imshow(clahe, cmap='bone')
ax[1].set_title("CLAHE Enhanced")

# 3. Grad-CAM (Simplified visualization)
ax[2].imshow(clahe, cmap='bone')
# Overlay heatmap logic here
ax[2].set_title(f"Grad-CAM: {disease_labels[pred_idx]}")

plt.show()    

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import cv2
import numpy as np

# Grad-CAM hook storage
activations = None
gradients = None

def forward_hook(module, input, output):
    global activations
    activations = output

def backward_hook(module, grad_in, grad_out):
    global gradients
    gradients = grad_out[0]

# Register hooks on the last convolutional block of ConvNeXt
# Adjust layer name depending on your model variant
target_layer = model.features[-1]  
target_layer.register_forward_hook(forward_hook)
target_layer.register_backward_hook(backward_hook)

def visualize_results(model, dataset, idx=0, device='cuda'):
    model.eval()
    img_tensor, labels = dataset[idx]
    input_batch = img_tensor.unsqueeze(0).to(device)

    # Forward pass
    output = model(input_batch)
    pred_idx = torch.argmax(output).item()
    target = output[:, pred_idx]

    # Backward pass
    model.zero_grad()
    target.backward()

    # Compute Grad-CAM
    weights = torch.mean(gradients, dim=(2, 3), keepdim=True)  # GAP over gradients
    cam = torch.sum(weights * activations, dim=1).squeeze()
    cam = torch.relu(cam)
    cam = cam.cpu().detach().numpy()
    cam = cv2.resize(cam, (img_tensor.shape[1], img_tensor.shape[2]))
    cam = cam - np.min(cam)
    cam = cam / np.max(cam)

    # Load original image
    img_path = all_image_paths[dataset.df.iloc[idx]['Image Index']]
    raw_img = cv2.imread(img_path)

    # CLAHE preprocessing
    gray = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(gray)

    # Heatmap overlay
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB), 0.5, heatmap, 0.5, 0)

    # Plot results
    fig, ax = plt.subplots(1, 3, figsize=(20, 10))

    ax[0].imshow(cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB))
    ax[0].set_title("Original X-Ray")

    ax[1].imshow(clahe, cmap='bone')
    ax[1].set_title("CLAHE Enhanced")

    ax[2].imshow(overlay)
    ax[2].set_title(f"Grad-CAM: {disease_labels[pred_idx]}")

    plt.show()
